In [0]:
# KPI 1: Total transations value
# KPI 2: Total transations count
# KPI 3: Deposit vs withbrawals
# KPI 4: monthly transaction value

In [0]:
%sql
-- KPI 1: Total Transaction Value (Completed transactions only)
DROP TABLE IF EXISTS dev.gold.kpi_total_transaction_value;
CREATE OR REPLACE VIEW dev.gold.kpi_total_transaction_value AS
SELECT 
    'Total Transaction Value' AS kpi_name,
    SUM(amount) AS total_value,
    COUNT(*) AS transaction_count,
    ROUND(AVG(amount), 2) AS avg_transaction_value,
    CURRENT_TIMESTAMP() AS calculated_at
FROM dev.silver.silver_transactions
WHERE transaction_status = 'Completed';

In [0]:
%sql
-- KPI 2: Total Transaction Count by Type and Status
CREATE OR REPLACE VIEW dev.gold.kpi_transaction_count AS
SELECT 
    transaction_type,
    transaction_status,
    COUNT(*) AS transaction_count,
    SUM(amount) AS total_amount,
    ROUND(AVG(amount), 2) AS avg_amount,
    MIN(transaction_date) AS first_transaction_date,
    MAX(transaction_date) AS last_transaction_date,
    CURRENT_TIMESTAMP() AS calculated_at
FROM dev.silver.silver_transactions
GROUP BY transaction_type, transaction_status
ORDER BY transaction_count DESC;

In [0]:
%sql
-- KPI 3: Deposit vs Withdrawals Analysis
CREATE OR REPLACE VIEW dev.gold.kpi_deposit_vs_withdrawal AS
WITH transaction_summary AS (
    SELECT 
        transaction_type,
        COUNT(*) AS transaction_count,
        SUM(amount) AS total_amount,
        ROUND(AVG(amount), 2) AS avg_amount
    FROM dev.silver.silver_transactions
    WHERE transaction_status = 'Completed'
      AND transaction_type IN ('Deposit', 'Withdrawal')
    GROUP BY transaction_type
)
SELECT 
    transaction_type,
    transaction_count,
    total_amount,
    avg_amount,
    ROUND(transaction_count * 100.0 / SUM(transaction_count) OVER (), 2) AS count_percentage,
    ROUND(total_amount * 100.0 / SUM(total_amount) OVER (), 2) AS amount_percentage,
    CURRENT_TIMESTAMP() AS calculated_at
FROM transaction_summary
ORDER BY total_amount DESC;

In [0]:
%sql
-- KPI 4: Monthly Transaction Value Trend
CREATE OR REPLACE VIEW dev.gold.kpi_monthly_transaction_value AS
SELECT 
    DATE_TRUNC('MONTH', transaction_date) AS transaction_month,
    YEAR(transaction_date) AS transaction_year,
    MONTH(transaction_date) AS transaction_month_num,
    transaction_type,
    COUNT(*) AS transaction_count,
    SUM(amount) AS total_amount,
    ROUND(AVG(amount), 2) AS avg_amount,
    MIN(amount) AS min_amount,
    MAX(amount) AS max_amount,
    CURRENT_TIMESTAMP() AS calculated_at
FROM dev.silver.silver_transactions
WHERE transaction_status = 'Completed'
GROUP BY 
    DATE_TRUNC('MONTH', transaction_date),
    YEAR(transaction_date),
    MONTH(transaction_date),
    transaction_type
ORDER BY transaction_month DESC, transaction_type;

In [0]:
%sql
-- ============================================================================
-- KPI SUMMARY DASHBOARD
-- ============================================================================

-- KPI 1: Overall Transaction Value Summary
SELECT 
    'KPI 1: Total Transaction Value' AS kpi_section,
    CONCAT('₹', FORMAT_NUMBER(total_value, 2)) AS total_value,
    FORMAT_NUMBER(transaction_count, 0) AS transaction_count,
    CONCAT('₹', FORMAT_NUMBER(avg_transaction_value, 2)) AS avg_transaction_value
FROM dev.gold.kpi_total_transaction_value;

-- KPI 2: Transaction Count by Type & Status (Top 5)
SELECT 
    'KPI 2: Transaction Count by Type' AS kpi_section,
    transaction_type,
    transaction_status,
    FORMAT_NUMBER(transaction_count, 0) AS transaction_count,
    CONCAT('₹', FORMAT_NUMBER(total_amount, 2)) AS total_amount
FROM dev.gold.kpi_transaction_count
ORDER BY transaction_count DESC
LIMIT 5;

-- KPI 3: Deposit vs Withdrawal Comparison
SELECT 
    'KPI 3: Deposit vs Withdrawal' AS kpi_section,
    transaction_type,
    FORMAT_NUMBER(transaction_count, 0) AS transaction_count,
    CONCAT(count_percentage, '%') AS count_percentage,
    CONCAT('₹', FORMAT_NUMBER(total_amount, 2)) AS total_amount,
    CONCAT(ROUND(amount_percentage, 2), '%') AS amount_percentage
FROM dev.gold.kpi_deposit_vs_withdrawal;

-- KPI 4: Monthly Transaction Trend (Last 6 Months)
SELECT 
    'KPI 4: Monthly Transaction Value' AS kpi_section,
    DATE_FORMAT(transaction_month, 'MMM yyyy') AS month,
    transaction_type,
    FORMAT_NUMBER(transaction_count, 0) AS transaction_count,
    CONCAT('₹', FORMAT_NUMBER(total_amount, 2)) AS total_amount,
    CONCAT('₹', FORMAT_NUMBER(avg_amount, 2)) AS avg_amount
FROM dev.gold.kpi_monthly_transaction_value
WHERE transaction_month >= DATE_SUB(CURRENT_DATE(), 180)
ORDER BY transaction_month DESC, transaction_type
LIMIT 24;